# 13. Debugging · Quality · Config

문제를 빠르게 찾고, 문서 품질을 자동 점검하고, App 기본 설정을 공유하는 도구.

## 1. Discovery

```python
app.help()                 # accessor 카테고리별 출력
repr(app)                  # 상태 요약 한 줄
```

실제 출력:


In [ ]:
import sys
sys.path.insert(0, '.')
from tutorial_helpers import run_and_show

In [ ]:
import io, contextlib
from hwpapi.core import App
buf = io.StringIO()
a = App(is_visible=False)
with contextlib.redirect_stdout(buf):
    print(repr(a))
    print()
    a.help()
try: a.quit()
except Exception: pass
print(buf.getvalue())

## 2. app.debug

### 2.1 state() — 현재 상태 dict 덤프

```python
app.debug.state()
# {
#   "cursor": {"doc_id": 1, "para": 15, "pos": 8},
#   "page": {"current": 3, "total": 10},
#   "selection": {"text": "Hello world", "length": 11},
#   "charshape_summary": {"fontsize": 1100, "bold": True, ...},
#   ...
# }

app.debug.print()          # 예쁘게 출력
```

### 2.2 timing(fn) — 함수 시간 측정

```python
r = app.debug.timing(app.insert_text, "테스트")
print(r)
# {'result': None, 'elapsed_ms': 12.34, 'exception': None, 'success': True}
```

### 2.3 trace() — COM 호출 로그

```python
with app.debug.trace():
    app.insert_text("hello")
# [trace 0] Run('InsertText') → True (0.8ms)
# [trace 1] Run('Cancel') → True (0.2ms)
# ...
```


## 3. app.lint() — 문서 품질 체크

callable accessor. `app.lint()` 호출 시 `LintReport` 반환.

```python
report = app.lint()
print(report.summary)
# Document lint — 7 issue(s)
#   Total: 42 paragraphs, 118 sentences, 8253 chars
#   Long sentences: 3
#   Empty paragraphs: 2

report.has_issues                  # True
report.long_sentences[:3]          # [(5, 103), (12, 95), (18, 87)]
report.empty_paragraphs            # [1, 17]

# 임계값 조정
report = app.lint(long_sentence_threshold=60)
```

실제 대상 문서 + lint 결과:


In [ ]:
run_and_show("lint_target", '''
app.set_charshape(bold=True, height=1600)
app.insert_text("품질 체크 대상 문서\n\n")
app.set_charshape(bold=False, height=1100)

# 긴 문장 1
app.insert_text("이 문장은 너무 길어서 린트 경고가 발생할 정도로 길게 작성되어야 하는데 정말 끝도 없이 이어지는 문장의 예시를 만들기 위해 계속 쓰고 있습니다.\n\n")

# 빈 문단 (여러 개의 줄바꿈)
app.insert_text("\n\n")

# 중복 공백
app.insert_text("이 문장에는  이중  공백이  있습니다.\n\n")

# 정상 문장
app.insert_text("정상적인 길이의 문장.\n")
''')

*위: `app.lint()` 가 감지할 수 있는 문제들이 포함된 테스트 문서 — 긴 문장, 빈 문단, 이중 공백 각각 존재.*

## 4. app.template — 문서 템플릿

```python
# 현재 설정 JSON 으로 저장
app.template.save("company_style.json")

# 다른 문서에 적용
app2 = App()
app2.open("new_doc.hwp")
app2.template.apply("company_style.json")
```

저장되는 JSON 구조:

```json
{
  "format": "hwpapi-template",
  "version": "1.0",
  "charshape": {
    "fontsize": 1100,
    "bold": true,
    "facename_hangul": "함초롬바탕"
  },
  "parashape": {
    "line_spacing": 160,
    "indent": 0
  }
}
```


## 5. app.config — App 기본 선호도

```python
app.config.default_font = "함초롬바탕"
app.config.default_size = 11
app.config.palette = {
    "primary": "#003366",
    "accent": "#FF6600",
}

# 일괄 업데이트
app.config.update(default_line_spacing=160)

# 디스크 저장 / 로드
app.config.save("~/.hwpapirc")
app2.config.load("~/.hwpapirc")
```


## 6. 실전 디버그 세션

```python
from hwpapi.core import App
app = App(is_visible=True)
app.open("problem.hwp")

# 1. 상태 파악
app.debug.print()

# 2. 문서 품질 체크
report = app.lint()
if report.has_issues:
    print(report.summary)
    for para_idx, length in sorted(report.long_sentences, key=lambda x: -x[1])[:3]:
        print(f"  para {para_idx}: {length}자")

# 3. 특정 작업 trace
with app.debug.trace():
    app.preset.striped_rows()
```
